# Week 3 – Exploratory Data Analysis and Visualization Strategy
**Dataset:** UCI Wine Quality – White Wine

This notebook implements the EDA workflow described in the report. It is designed to be reproducible and does not overwrite the raw dataset.

In [ ]:
# Install once if needed:
# %pip install pandas numpy matplotlib seaborn scipy plotly

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", None)
sns.set_theme()

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
df = pd.read_csv(URL, sep=";")

print("Shape:", df.shape)
display(df.head())

## 1. Structural and data-quality audit

In [ ]:
print(df.info())
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDescriptive statistics:")
display(df.describe().T)

In [ ]:
# Unique-value and range review
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unique_values": df.nunique(),
    "min": df.min(numeric_only=True),
    "max": df.max(numeric_only=True),
    "missing": df.isna().sum()
})
display(audit)

## 2. Descriptive statistics and skewness

In [ ]:
summary = df.describe().T
summary["median"] = df.median(numeric_only=True)
summary["IQR"] = df.quantile(0.75) - df.quantile(0.25)
summary["skewness"] = df.skew(numeric_only=True)
display(summary[["count","mean","median","std","min","25%","50%","75%","max","IQR","skewness"]])

## 3. Target distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x="quality")
plt.title("Distribution of Wine Quality Scores")
plt.xlabel("Quality score")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.savefig("outputs/01_quality_distribution.png", dpi=200)
plt.show()

## 4. Univariate distributions

In [ ]:
numeric_cols = df.columns.tolist()
n = len(numeric_cols)
fig, axes = plt.subplots(4, 3, figsize=(15, 16))
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)

for ax in axes[len(numeric_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("outputs/02_univariate_distributions.png", dpi=200)
plt.show()

## 5. Box plots for outlier screening

In [ ]:
plt.figure(figsize=(14,7))
sns.boxplot(data=df, orient="h")
plt.title("Box Plots of Numerical Variables")
plt.tight_layout()
plt.savefig("outputs/03_boxplots.png", dpi=200)
plt.show()

## 6. Correlation analysis

In [ ]:
corr = df.corr(numeric_only=True, method="pearson")

plt.figure(figsize=(12,9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Pearson Correlation Heatmap")
plt.tight_layout()
plt.savefig("outputs/04_correlation_heatmap.png", dpi=200)
plt.show()

print("Correlations with quality:")
display(corr["quality"].sort_values(ascending=False).to_frame("pearson_corr_with_quality"))

## 7. Spearman correlations with ordinal quality

In [ ]:
spearman = df.corr(numeric_only=True, method="spearman")["quality"].sort_values(ascending=False)
display(spearman.to_frame("spearman_corr_with_quality"))

## 8. Relationship plots

In [ ]:
top_candidates = ["alcohol", "density", "volatile acidity", "chlorides", "sulphates"]

fig, axes = plt.subplots(2, 3, figsize=(16,9))
axes = axes.flatten()

for ax, col in zip(axes, top_candidates):
    sns.scatterplot(data=df, x=col, y="quality", alpha=0.35, ax=ax)
    sns.regplot(data=df, x=col, y="quality", scatter=False, ax=ax, color="black")
    ax.set_title(f"{col} vs quality")

axes[-1].axis("off")
plt.tight_layout()
plt.savefig("outputs/05_quality_relationships.png", dpi=200)
plt.show()

## 9. Group comparison

In [ ]:
# Group quality scores into interpretable bands for visualization only.
def quality_group(x):
    if x <= 4:
        return "Low (<=4)"
    elif x <= 6:
        return "Medium (5-6)"
    return "High (>=7)"

df["quality_group"] = df["quality"].apply(quality_group)

plt.figure(figsize=(12,6))
sns.boxplot(data=df, x="quality_group", y="alcohol", order=["Low (<=4)", "Medium (5-6)", "High (>=7)"])
plt.title("Alcohol Distribution by Quality Group")
plt.tight_layout()
plt.savefig("outputs/06_group_comparison_alcohol.png", dpi=200)
plt.show()

## 10. IQR-based outlier counts

In [ ]:
outlier_rows = []

for col in df.select_dtypes(include=np.number).columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_rows.append([col, q1, q3, iqr, lower, upper, int(count)])

outlier_summary = pd.DataFrame(
    outlier_rows,
    columns=["variable","Q1","Q3","IQR","lower_fence","upper_fence","outlier_count"]
).sort_values("outlier_count", ascending=False)

display(outlier_summary)

## 11. Interpretation checklist

When reviewing the generated outputs:

1. Report the shape, data types, missing values and duplicates.
2. Describe the quality-score distribution and any imbalance.
3. Identify strongly skewed variables.
4. Compare Pearson and Spearman associations with quality.
5. Investigate strong predictor-predictor correlations for redundancy.
6. Treat IQR flags as candidates for investigation, not automatic deletions.
7. Record the strongest evidence-backed patterns and their limitations.
8. Do not claim causality from correlation.

In [ ]:
# Optional: save a compact EDA summary for the report
quality_corr = pd.DataFrame({
    "pearson": df.drop(columns="quality_group").corr(numeric_only=True)["quality"],
    "spearman": df.drop(columns="quality_group").corr(numeric_only=True, method="spearman")["quality"]
}).sort_values("spearman", ascending=False)

quality_corr.to_csv("outputs/quality_correlations.csv")
outlier_summary.to_csv("outputs/outlier_summary.csv", index=False)
print("Saved summary files to outputs/.")